# LeetCode 2023 — Number of Pairs of Strings With Concatenation Equal to Target bằng Python

Link đề: https://leetcode.com/problems/number-of-pairs-of-strings-with-concatenation-equal-to-target/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def numOfPairs(self, nums: List[str], target: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Đếm **cặp có thứ tự** `(i, j)` với `i ≠ j` — phân biệt với cặp không thứ tự `i < j`.
2. Nhận ra `a + b == target` ⇔ `a` là **tiền tố**, `b` là **hậu tố** của `target` và `len(a) + len(b) == len(target)`.
3. Đổi góc nhìn: thay vì duyệt cặp phần tử, duyệt **vị trí cắt** của `target` (≤ 99 vị trí) và dùng **bảng đếm** `Counter`.
4. Xử lý trường hợp tiền tố **bằng** hậu tố: số cặp là `c·(c−1)` (không ghép phần tử với chính nó).
5. Kiểm thử đối chiếu brute force trên dữ liệu ngẫu nhiên có nhiều chuỗi lặp (seed cố định).

## 2. Đề bài & ví dụ

Cho mảng chuỗi chữ số `nums` và chuỗi chữ số `target`. Đếm số cặp chỉ số `(i, j)` với `i ≠ j` sao cho `nums[i] + nums[j] == target` (ghép chuỗi).

**Ví dụ 1:** `nums = ["777","7","77","77"]`, `target = "7777"` → `4`: `(0,1)`, `(1,0)`, `(2,3)`, `(3,2)`.

**Ví dụ 2:** `nums = ["123","4","12","34"]`, `target = "1234"` → `2`: `(0,1)`, `(2,3)`.

**Ví dụ 3:** `nums = ["1","1","1"]`, `target = "11"` → `6`: mọi cặp có thứ tự của 3 chỉ số.

**Ràng buộc:** `2 ≤ len(nums) ≤ 100`, `1 ≤ len(nums[i]) ≤ 100`, `2 ≤ len(target) ≤ 100`, chỉ gồm chữ số, không có số 0 đứng đầu.

In [1]:
import random
import time
from collections import Counter
from itertools import permutations
from typing import List

examples = [(["777", "7", "77", "77"], "7777", 4), (["123", "4", "12", "34"], "1234", 2), (["1", "1", "1"], "11", 6)]
for nums, target, exp in examples:
    print(nums, repr(target), "-> kỳ vọng", exp)

['777', '7', '77', '77'] '7777' -> kỳ vọng 4
['123', '4', '12', '34'] '1234' -> kỳ vọng 2
['1', '1', '1'] '11' -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1 Cặp có thứ tự vs. không thứ tự

Với `n` phần tử: số cặp có thứ tự `(i, j), i ≠ j` là `n·(n−1)`; số cặp không thứ tự `i < j` là `n·(n−1)/2`. Ở bài này `(0,1)` và `(1,0)` là **hai** cặp khác nhau.

In [2]:
n = 3
print("có thứ tự:", list(permutations(range(n), 2)), "->", n * (n - 1))
print("không thứ tự:", [(i, j) for i in range(n) for j in range(i + 1, n)], "->", n * (n - 1) // 2)

có thứ tự: [(0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1)] -> 6
không thứ tự: [(0, 1), (0, 2), (1, 2)] -> 3


### 3.2 `startswith`, `endswith` và cắt chuỗi

`t.startswith(a)` kiểm `a` là tiền tố; `t.endswith(b)` kiểm hậu tố. `t[:k]` là `k` ký tự đầu, `t[k:]` là phần còn lại. Với `0 < k < len(t)`, `(t[:k], t[k:])` liệt kê mọi cách cắt `t` thành hai phần khác rỗng.

In [3]:
t = "1234"
print([(t[:k], t[k:]) for k in range(1, len(t))])
print(t.startswith("12"), t.endswith("34"), t.startswith("23"))

[('1', '234'), ('12', '34'), ('123', '4')]
True True False


### 3.3 `Counter` — đếm số lần xuất hiện của mỗi chuỗi

`cnt = Counter(nums)`; `cnt["77"]` cho số lần `"77"` xuất hiện; khóa không có trả `0`.

In [4]:
cnt = Counter(["777", "7", "77", "77"])
print(cnt, "| cnt['77'] =", cnt["77"], "| cnt['x'] =", cnt["x"])

Counter({'77': 2, '777': 1, '7': 1}) | cnt['77'] = 2 | cnt['x'] = 0


### 3.4 Quy tắc nhân khi đếm cặp

Nếu có `a` cách chọn phần tử trái và `b` cách chọn phần tử phải, và hai lựa chọn độc lập (chuỗi khác nhau ⇒ chỉ số khác nhau), số cặp là `a·b`. Nếu hai phần **giống nhau** (cùng chuỗi), chọn trái có `c` cách, chọn phải chỉ còn `c − 1` cách (khác chỉ số) ⇒ `c·(c−1)`.

In [5]:
c = 3
print("cùng chuỗi, c = 3:", c * (c - 1), "| khác chuỗi, a = 2, b = 3:", 2 * 3)

cùng chuỗi, c = 3: 6 | khác chuỗi, a = 2, b = 3: 6


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1, `target = "7777"`. Các cách cắt:

| k | tiền tố | hậu tố | cnt[tiền tố] | cnt[hậu tố] | số cặp |
|---|---|---|---|---|---|
| 1 | "7" | "777" | 1 | 1 | 1·1 = 1 |
| 2 | "77" | "77" | 2 | 2 | **2·1 = 2** (cùng chuỗi) |
| 3 | "777" | "7" | 1 | 1 | 1 |

Tổng `4` ✓. Mỗi cặp `(i, j)` hợp lệ ứng với **đúng một** cách cắt `k = len(nums[i])`, nên cộng theo `k` không đếm trùng.

In [6]:
nums, target = examples[0][:2]
cnt = Counter(nums)
for k in range(1, len(target)):
    p, s = target[:k], target[k:]
    ways = cnt[p] * (cnt[s] - (1 if p == s else 0))
    print(f"k={k}: {p!r} + {s!r}: cnt={cnt[p]}, {cnt[s]} -> {ways}")

k=1: '7' + '777': cnt=1, 1 -> 1
k=2: '77' + '77': cnt=2, 2 -> 2
k=3: '777' + '7': cnt=1, 1 -> 1


## 5. Cách 1 — Brute Force / baseline: mọi cặp có thứ tự, ghép chuỗi

**Ý tưởng:** hai vòng lặp `i`, `j`, bỏ `i == j`, so `nums[i] + nums[j] == target`.

In [7]:
def pairs_brute(nums, target):
    n = len(nums)
    ans = 0
    for i in range(n):
        for j in range(n):
            if i != j and nums[i] + nums[j] == target:
                ans += 1
    return ans

for nums, target, exp in examples:
    got = pairs_brute(nums, target); print(got, "kỳ vọng", exp); assert got == exp

4 kỳ vọng 4
2 kỳ vọng 2
6 kỳ vọng 6


**Độ phức tạp:** `n·(n−1)` cặp, mỗi cặp tạo chuỗi dài tới `200` và so sánh `O(L)` ⇒ `O(n²·L)` ≈ `10⁴ · 200 = 2·10⁶` — **vẫn chạy được** với ràng buộc nhỏ của đề. Space `O(L)` cho chuỗi tạm.

**Nút thắt (khi dữ liệu lớn hơn):** ta xét lại cùng cặp chuỗi nhiều lần nếu `nums` có chuỗi lặp, và phải duyệt mọi cặp dù hầu hết không phải tiền tố/hậu tố của `target`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** `a + b == target` ⇔ tồn tại `k` (`1 ≤ k < len(target)`) với `a == target[:k]` và `b == target[k:]`. Có tối đa `L − 1 ≤ 99` cách cắt, còn số cặp phần tử có thể tới `9900` (hoặc rất lớn nếu `n` lớn).

**Quy tắc "nếu… thì…":**
- Với mỗi `k`: `p = target[:k]`, `s = target[k:]`.
- **Nếu** `p != s` **thì** cộng `cnt[p] · cnt[s]` (chỉ số chắc chắn khác nhau vì chuỗi khác nhau).
- **Nếu** `p == s` **thì** cộng `cnt[p] · (cnt[p] − 1)`.

**Cách nhìn theo phần tử (một lượt):** với mỗi `x = nums[i]` là tiền tố của `target`, phần cần ghép là `rest = target[len(x):]`; số `j` hợp lệ là `cnt[rest]`, trừ `1` nếu `rest == x` (không được chọn chính `i`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Duyệt cặp nhưng không tạo chuỗi ghép

Kiểm `len(a) + len(b) == len(target)`, `target.startswith(a)`, `target.endswith(b)` — tránh cấp phát chuỗi mới. Vẫn `O(n²)` cặp nhưng nhanh hơn hằng số và **loại sớm** theo độ dài.

In [8]:
def pairs_no_concat(nums, target):
    L = len(target); ans = 0
    pre = [target.startswith(x) for x in nums]        # tiền tính: x có là tiền tố?
    suf = [target.endswith(x) for x in nums]          # x có là hậu tố?
    for i, a in enumerate(nums):
        if not pre[i]:
            continue
        for j, b in enumerate(nums):
            if i != j and suf[j] and len(a) + len(b) == L:
                ans += 1
    return ans

for nums, target, exp in examples:
    assert pairs_no_concat(nums, target) == exp
print([pairs_no_concat(n, t) for n, t, _ in examples])

[4, 2, 6]


**Độ phức tạp Cách 2:** tiền tính `O(n·L)`, vòng cặp `O(n²)` với mỗi bước `O(1)` ⇒ `O(n·L + n²)`.

### 7.2 Cách 3 — Duyệt vị trí cắt + `Counter` (bản nộp chính)

**Giải thích từng đoạn:**
1. `cnt = Counter(nums)`.
2. Với `k = 1..L−1`: `p, s = target[:k], target[k:]`.
3. Cộng `cnt[p] * cnt[s]` nếu `p != s`, ngược lại `cnt[p] * (cnt[p] − 1)`.

In [9]:
def pairs_split(nums, target):
    cnt = Counter(nums)
    ans = 0
    for k in range(1, len(target)):
        p, s = target[:k], target[k:]
        if p != s:
            ans += cnt[p] * cnt[s]
        else:
            ans += cnt[p] * (cnt[p] - 1)
    return ans

for nums, target, exp in examples:
    assert pairs_split(nums, target) == exp
print([pairs_split(n, t) for n, t, _ in examples])

[4, 2, 6]


In [10]:
class SolutionSplit:
    def numOfPairs(self, nums: List[str], target: str) -> int:
        cnt = Counter(nums)
        ans = 0
        for k in range(1, len(target)):
            p, s = target[:k], target[k:]
            ans += cnt[p] * (cnt[s] - (p == s))
        return ans

print([SolutionSplit().numOfPairs(n, t) for n, t, _ in examples])

[4, 2, 6]


**Độ phức tạp Cách 3:** dựng `Counter` `O(n·L)` (băm từng chuỗi), duyệt `L − 1` vị trí cắt, mỗi vị trí tạo/băm hai chuỗi `O(L)` ⇒ `O(n·L + L²)`. Space `O(n·L)`.

Mẹo: `cnt[s] - (p == s)` dùng việc `True == 1`, `False == 0` trong Python.

### 7.3 Cách 4 — Một lượt theo phần tử + `Counter`

Với mỗi chuỗi `x` trong `nums`: nếu `target.startswith(x)` và `len(x) < L`, phần còn lại `rest = target[len(x):]`; số chỉ số `j` hợp lệ là `cnt[rest] − (rest == x)`.

In [11]:
def pairs_per_element(nums, target):
    cnt = Counter(nums); L = len(target); ans = 0
    for x in nums:
        if len(x) < L and target.startswith(x):
            rest = target[len(x):]
            ans += cnt[rest] - (rest == x)
    return ans

for nums, target, exp in examples:
    assert pairs_per_element(nums, target) == exp
print([pairs_per_element(n, t) for n, t, _ in examples])

[4, 2, 6]


**Độ phức tạp Cách 4:** `O(n·L)`. Space `O(n·L)`. Cách này tự nhiên khi bạn nghĩ "với mỗi phần tử đứng trước, cần phần tử đứng sau là gì?" — giống bài Two Sum.

### 7.4 Cách 5 — Một lượt "trực tuyến" như Two Sum (không cần trừ chính nó)

Duyệt từ trái sang phải, chỉ ghép phần tử hiện tại với các phần tử **đã gặp**: phần tử hiện tại có thể đứng **sau** (cần tiền tố đã gặp) hoặc đứng **trước** (cần hậu tố đã gặp). Vì chỉ ghép với phần tử đã gặp, không bao giờ ghép `i` với chính nó.

In [12]:
def pairs_online(nums, target):
    seen = Counter(); L = len(target); ans = 0
    for x in nums:
        m = len(x)
        if m < L:
            if target.endswith(x):          # x đứng sau, cần tiền tố target[:L-m] đã gặp
                ans += seen[target[:L - m]]
            if target.startswith(x):        # x đứng trước, cần hậu tố target[m:] đã gặp
                ans += seen[target[m:]]
        seen[x] += 1
    return ans

for nums, target, exp in examples:
    assert pairs_online(nums, target) == exp
print([pairs_online(n, t) for n, t, _ in examples])

[4, 2, 6]


**Độ phức tạp Cách 5:** `O(n·L)`, Space `O(n·L)`.

## 8. TRACE từng bước

Trace Cách 5 trên ví dụ 3 `nums = ["1","1","1"]`, `target = "11"`:

| i | x | là hậu tố? cần tiền tố | seen[tiền tố] | là tiền tố? cần hậu tố | seen[hậu tố] | ans | seen sau |
|---|---|---|---|---|---|---|---|
| 0 | "1" | ✓ "1" | 0 | ✓ "1" | 0 | 0 | {"1":1} |
| 1 | "1" | ✓ "1" | 1 | ✓ "1" | 1 | 2 | {"1":2} |
| 2 | "1" | ✓ "1" | 2 | ✓ "1" | 2 | 6 | {"1":3} |

In [13]:
def trace_online(nums, target):
    seen = Counter(); L = len(target); ans = 0
    for i, x in enumerate(nums):
        m = len(x); add1 = add2 = 0
        if m < L and target.endswith(x):
            add1 = seen[target[:L - m]]
        if m < L and target.startswith(x):
            add2 = seen[target[m:]]
        ans += add1 + add2
        print(f"i={i} x={x!r}: làm vế sau +{add1} (cần {target[:L - m]!r}), "
              f"làm vế trước +{add2} (cần {target[m:]!r}) -> ans={ans}")
        seen[x] += 1
    return ans

print("Đáp án:", trace_online(*examples[2][:2]))
print()
print("Đáp án:", trace_online(*examples[0][:2]))

i=0 x='1': làm vế sau +0 (cần '1'), làm vế trước +0 (cần '1') -> ans=0
i=1 x='1': làm vế sau +1 (cần '1'), làm vế trước +1 (cần '1') -> ans=2
i=2 x='1': làm vế sau +2 (cần '1'), làm vế trước +2 (cần '1') -> ans=6
Đáp án: 6

i=0 x='777': làm vế sau +0 (cần '7'), làm vế trước +0 (cần '7') -> ans=0
i=1 x='7': làm vế sau +1 (cần '777'), làm vế trước +1 (cần '777') -> ans=2
i=2 x='77': làm vế sau +0 (cần '77'), làm vế trước +0 (cần '77') -> ans=2
i=3 x='77': làm vế sau +1 (cần '77'), làm vế trước +1 (cần '77') -> ans=4
Đáp án: 4


In [14]:
def trace_split(nums, target):
    cnt = Counter(nums); ans = 0
    print("cnt =", dict(cnt))
    for k in range(1, len(target)):
        p, s = target[:k], target[k:]
        add = cnt[p] * (cnt[s] - (p == s))
        ans += add
        print(f"k={k}: {p!r} | {s!r} {'(bằng nhau)' if p == s else ''}: +{add} -> ans={ans}")
    return ans

print("Đáp án:", trace_split(*examples[1][:2]))

cnt = {'123': 1, '4': 1, '12': 1, '34': 1}
k=1: '1' | '234' : +0 -> ans=0
k=2: '12' | '34' : +1 -> ans=1
k=3: '123' | '4' : +1 -> ans=2
Đáp án: 2


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def numOfPairs(self, nums: List[str], target: str) -> int:
        cnt = Counter(nums)
        ans = 0
        for k in range(1, len(target)):          # cắt target thành p | s
            p, s = target[:k], target[k:]
            if p != s:
                ans += cnt[p] * cnt[s]
            else:
                ans += cnt[p] * (cnt[p] - 1)     # không ghép một chỉ số với chính nó
        return ans

for nums, target, exp in examples:
    assert Solution().numOfPairs(nums, target) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Song ánh cặp ↔ (vị trí cắt, chỉ số).** Mỗi cặp hợp lệ `(i, j)` có `nums[i] + nums[j] = target`, nên `nums[i] = target[:k]`, `nums[j] = target[k:]` với `k = len(nums[i])`, và `1 ≤ k ≤ L − 1` (vì mọi chuỗi khác rỗng). Ngược lại, với mỗi `k` và mỗi cặp chỉ số `i ≠ j` có `nums[i] = target[:k]`, `nums[j] = target[k:]`, ta được một cặp hợp lệ. Các `k` khác nhau cho các tập cặp **rời nhau** (vì `len(nums[i])` khác nhau).

**Đếm cho một `k`.** Số `i` có `nums[i] = p` là `cnt[p]`, số `j` có `nums[j] = s` là `cnt[s]`. Nếu `p ≠ s` thì mọi cặp như vậy tự động có `i ≠ j` ⇒ `cnt[p]·cnt[s]`. Nếu `p = s` thì phải loại `cnt[p]` cặp có `i = j` ⇒ `cnt[p]² − cnt[p] = cnt[p]·(cnt[p] − 1)`.

Cộng theo mọi `k` cho đúng tổng số cặp.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `L = len(target)`, `ℓ` = độ dài tối đa của `nums[i]`.

| Cách | Time | Space |
|---|---|---|
| 1. Ghép mọi cặp | `O(n²·(ℓ + L))` | `O(L)` |
| 2. Cặp + tiền tính tiền/hậu tố | `O(n·L + n²)` | `O(n)` |
| 3. Cắt target + Counter | `O(n·ℓ + L²)` | `O(n·ℓ)` |
| 4. Theo phần tử + Counter | `O(n·(ℓ + L))` | `O(n·ℓ)` |
| 5. Trực tuyến (Two Sum) | `O(n·(ℓ + L))` | `O(n·ℓ)` |

Ràng buộc gốc nhỏ nên mọi cách đều qua; khác biệt rõ khi `n` lớn (ví dụ `n = 10⁴`):

In [16]:
rng = random.Random(2023)
target = "".join(rng.choice("12") for _ in range(40))
big = [target[:k] if rng.random() < 0.5 else target[k:] for k in (rng.randint(1, 39) for _ in range(3000))]
for name, fn in [("brute", pairs_brute), ("no_concat", pairs_no_concat), ("split", pairs_split),
                 ("per_elem", pairs_per_element), ("online", pairs_online)]:
    t = time.perf_counter(); r = fn(big, target)
    print(f"{name:9s}: {r} ({time.perf_counter() - t:.3f}s)")

brute    : 60321 (0.522s)


no_concat: 60321 (0.221s)
split    : 60321 (0.000s)
per_elem : 60321 (0.000s)
online   : 60321 (0.001s)


## 12. Bộ test

Ví dụ đề; biên (tiền tố bằng hậu tố, không có cặp nào, chuỗi dài bằng `target`, chuỗi dài hơn `target`, `n = 2`); đối chiếu brute force trên dữ liệu ngẫu nhiên (bảng chữ nhỏ để có nhiều trùng lặp; seed cố định).

In [17]:
fns = {
    "brute": pairs_brute, "no_concat": pairs_no_concat, "split": pairs_split,
    "per_elem": pairs_per_element, "online": pairs_online,
    "Solution": lambda n, t: Solution().numOfPairs(n, t), "SolutionSplit": lambda n, t: SolutionSplit().numOfPairs(n, t),
}
edge = [
    (["1", "2"], "12", 1), (["1", "2"], "21", 1), (["1", "2"], "11", 0),
    (["12", "1", "2"], "12", 1),                # "12" dài bằng target: không ghép được
    (["123", "4"], "12", 0),                    # chuỗi dài hơn target
    (["11", "11", "11", "11"], "1111", 12),     # 4 * 3
    (["1", "11", "111"], "111", 2),
]
for nums, target, exp in [(n, t, e) for n, t, e in examples] + edge:
    for name, fn in fns.items():
        assert fn(nums, target) == exp, (name, nums, target, fn(nums, target))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [18]:
rng = random.Random(42)
for _ in range(1000):
    alpha = rng.choice(["1", "12", "123"])
    target = "".join(rng.choice(alpha) for _ in range(rng.randint(2, 7)))
    nums = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(2, 10))]
    if rng.random() < 0.5:                      # chèn thêm các mảnh của target
        nums += [target[:k] for k in range(1, len(target)) if rng.random() < 0.4]
        nums += [target[k:] for k in range(1, len(target)) if rng.random() < 0.4]
    want = pairs_brute(nums, target)
    for name, fn in fns.items():
        assert fn(nums, target) == want, (name, nums, target)
print("1000 test ngẫu nhiên: mọi cách khớp brute force")

1000 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm cặp `i < j` (không thứ tự)

Đề đếm cặp **có thứ tự**: `"777" + "7"` và `"7" + "777"` là hai cặp khác nhau (ở đây cả hai đều đúng). Đếm `i < j` và chỉ thử một chiều ghép sẽ thiếu.

In [19]:
def wrong_unordered(nums, target):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) if nums[i] + nums[j] == target)   # SAI

nums, target, exp = examples[0]
print("sai:", wrong_unordered(nums, target), "| đúng:", exp)

sai: 2 | đúng: 4


In [20]:
print("sửa (mọi i != j):", pairs_brute(nums, target))

sửa (mọi i != j): 4


### Lỗi 2 — Quên trừ cặp `i = j` khi tiền tố bằng hậu tố

`cnt[p] * cnt[s]` với `p == s` sẽ đếm cả cặp một chỉ số ghép với chính nó.

In [21]:
def wrong_self_pair(nums, target):
    cnt = Counter(nums)
    return sum(cnt[target[:k]] * cnt[target[k:]] for k in range(1, len(target)))   # SAI khi p == s

nums, target, exp = examples[2]
print("sai:", wrong_self_pair(nums, target), "| đúng:", exp)

sai: 9 | đúng: 6


In [22]:
print("sửa:", pairs_split(nums, target))

sửa: 6


### Lỗi 3 — Kiểm `startswith(a)` và `endswith(b)` nhưng quên điều kiện độ dài

`a = "1"`, `b = "1"`, `target = "121"`: `a` là tiền tố, `b` là hậu tố, nhưng `"1" + "1" = "11" ≠ "121"`.

In [23]:
def wrong_no_length(nums, target):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(n)
               if i != j and target.startswith(nums[i]) and target.endswith(nums[j]))   # SAI: thiếu độ dài

nums, target = ["1", "1"], "121"
print("sai:", wrong_no_length(nums, target), "| đúng:", pairs_brute(nums, target))

sai: 2 | đúng: 0


In [24]:
print("sửa (thêm len(a) + len(b) == len(target)):", pairs_no_concat(nums, target))

sửa (thêm len(a) + len(b) == len(target)): 0


### Lỗi 4 — Dùng `set(nums)` thay cho `Counter` (mất số lần lặp)

Ví dụ 1 có hai chuỗi `"77"`; dùng `set` thì chỉ còn một ⇒ mất các cặp `(2,3)`, `(3,2)`.

In [25]:
def wrong_set(nums, target):
    s = set(nums)
    return sum(1 for k in range(1, len(target)) if target[:k] in s and target[k:] in s and target[:k] != target[k:])  # SAI

nums, target, exp = examples[0]
print("sai:", wrong_set(nums, target), "| đúng:", exp)

sai: 2 | đúng: 4


In [26]:
print("sửa (Counter):", Solution().numOfPairs(nums, target))

sửa (Counter): 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm cặp sao cho f(a, b) = mục tiêu"** ⇒ cố định một vế, tra vế còn lại trong `Counter` (giống Two Sum).
- **Ghép chuỗi bằng `target`** ⇒ vế trái là **tiền tố**, vế phải là **hậu tố** ⇒ chỉ có `L − 1` cách cắt.
- Cặp **có thứ tự** + hai vế giống nhau ⇒ `c·(c−1)`; cặp **không thứ tự** ⇒ `c·(c−1)/2`.
- Duyệt "trực tuyến" (chỉ ghép với phần tử đã gặp) tự động tránh ghép với chính mình.
- Ràng buộc nhỏ (`n ≤ 100`) ⇒ brute force cũng qua; nhưng biết cách `Counter` để dùng cho biến thể lớn.

## 15. Biến thể của bài

1. **LeetCode 1 — Two Sum:** cùng tư duy "vế còn lại cần là gì".
2. **LeetCode 336 — Palindrome Pairs (Hard):** đếm cặp có `a + b` là palindrome — cắt mỗi chuỗi và tra phần đảo ngược.
3. **Đếm bộ ba** `(i, j, k)` đôi một khác nhau với `nums[i] + nums[j] + nums[k] == target` (bài tập 5).
4. **Ghép với ký tự đại diện** hoặc **ghép số** (`int(a + b) == x`) — vẫn là cắt `target`.
5. **Cặp không thứ tự** (`i < j`, cả hai chiều ghép đều tính): công thức khác một chút.

In [27]:
def pairs_unordered_any_order(nums, target):
    """Số cặp i < j sao cho nums[i]+nums[j] == target HOẶC nums[j]+nums[i] == target (biến thể tự dựng)."""
    n = len(nums); total = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] + nums[j] == target or nums[j] + nums[i] == target:
                total += 1
    return total

def pairs_unordered_fast(nums, target):
    cnt = Counter(nums); total = 0
    for k in range(1, len(target)):
        p, s = target[:k], target[k:]
        if p == s:
            total += cnt[p] * (cnt[p] - 1) // 2
        elif (p, s) < (s, p) or target != s + p:      # tránh đếm hai lần cặp {p, s} khi s+p cũng bằng target
            total += cnt[p] * cnt[s]
    return total

rng = random.Random(15)
for _ in range(500):
    target = "".join(rng.choice("12") for _ in range(rng.randint(2, 6)))
    nums = ["".join(rng.choice("12") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(2, 9))]
    assert pairs_unordered_fast(nums, target) == pairs_unordered_any_order(nums, target)
print("Biến thể cặp không thứ tự khớp brute force; ['1','1','1'], '11' ->", pairs_unordered_fast(["1", "1", "1"], "11"))

Biến thể cặp không thứ tự khớp brute force; ['1','1','1'], '11' -> 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ghép mọi cặp | brute force | `O(n²·L)` | `O(L)` | ràng buộc nhỏ, kiểm tra |
| 2. Cặp + tiền/hậu tố | không tạo chuỗi | `O(n·L + n²)` | `O(n)` | khi muốn tránh cấp phát |
| 3. Cắt target + Counter | `L − 1` vị trí cắt | `O(n·ℓ + L²)` | `O(n·ℓ)` | **nộp bài**, `n` lớn |
| 4. Theo phần tử | tra phần còn lại | `O(n·L)` | `O(n·ℓ)` | quen kiểu Two Sum |
| 5. Trực tuyến | ghép với phần tử đã gặp | `O(n·L)` | `O(n·ℓ)` | dữ liệu dạng luồng |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `nums = ["1","11","1","11"]`, `target = "111"`.
2. **Tính tay:** `nums = ["12","3","1","23","1"]`, `target = "123"`: lập bảng theo vị trí cắt.
3. **Điền TODO:** `list_pairs(nums, target)` trả về **danh sách** các cặp `(i, j)` hợp lệ trong `O(n·L + số cặp)`.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `count_triples(nums, target)` — số bộ ba có thứ tự `(i, j, k)` **đôi một khác nhau** với `nums[i] + nums[j] + nums[k] == target`. Gợi ý: duyệt hai vị trí cắt; kiểm bằng brute force.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def list_pairs(nums, target):
    res = []
    # TODO: gom chỉ số theo chuỗi: idx[s] = [i...]; với mỗi k ghép idx[p] x idx[s], bỏ i == j
    return res

In [29]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_num_of_pairs(nums, target):
    # TODO
    pass

In [30]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def count_triples(nums, target):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Cắt `"1"|"11"` và `"11"|"1"`. (2) `cnt["1"] = 2`, `cnt["11"] = 2`. (3) `2·2 + 2·2 = 8`.

**Bài 2.** (1) `k = 1`: `"1"|"23"` ⇒ `cnt = 2, 1` ⇒ `2`. (2) `k = 2`: `"12"|"3"` ⇒ `1·1 = 1`. (3) Tổng `3`.

**Bài 3.** (1) `idx = defaultdict(list)`. (2) Với mỗi `k`, duyệt `i in idx[p]`, `j in idx[s]`. (3) Bỏ `i == j`.

**Bài 4.** (1) `Counter(nums)`. (2) `k` từ 1 tới `L − 1`. (3) `cnt[p] * (cnt[s] − (p == s))`.

**Bài 5.** (1) Hai vị trí cắt `1 ≤ a < b ≤ L − 1` cho ba mảnh `x, y, z`. (2) Số cách chọn ba chỉ số đôi một khác nhau: đếm theo đa tập `{x, y, z}` — nếu cả ba khác nhau: `cx·cy·cz`; nếu `x = y ≠ z`: `cx·(cx−1)·cz`; ... nếu cả ba bằng nhau: `c·(c−1)·(c−2)`. (3) Tổng quát: duyệt lần lượt, mỗi lần dùng một chuỗi thì "trừ tạm" `cnt` của chuỗi đó đi 1.

## 19. Đáp án tham khảo

In [31]:
# Bài 1, 2
assert Solution().numOfPairs(["1", "11", "1", "11"], "111") == 8
assert Solution().numOfPairs(["12", "3", "1", "23", "1"], "123") == 3
print("Bài 1: 8 | Bài 2: 3")
trace_split(["12", "3", "1", "23", "1"], "123")

Bài 1: 8 | Bài 2: 3
cnt = {'12': 1, '3': 1, '1': 2, '23': 1}
k=1: '1' | '23' : +2 -> ans=2
k=2: '12' | '3' : +1 -> ans=3


3

In [32]:
# Bài 3, 4
from collections import defaultdict
def list_pairs_ans(nums, target):
    idx = defaultdict(list)
    for i, x in enumerate(nums): idx[x].append(i)
    res = []
    for k in range(1, len(target)):
        for i in idx.get(target[:k], []):
            for j in idx.get(target[k:], []):
                if i != j: res.append((i, j))
    return res

my_num_of_pairs_ans = pairs_split
rng = random.Random(3)
for _ in range(300):
    target = "".join(rng.choice("12") for _ in range(rng.randint(2, 6)))
    nums = ["".join(rng.choice("12") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(2, 8))]
    brute = sorted((i, j) for i in range(len(nums)) for j in range(len(nums)) if i != j and nums[i] + nums[j] == target)
    assert sorted(list_pairs_ans(nums, target)) == brute and my_num_of_pairs_ans(nums, target) == len(brute)
print("Bài 3, 4: OK; ví dụ 1:", sorted(list_pairs_ans(*examples[0][:2])))

Bài 3, 4: OK; ví dụ 1: [(0, 1), (1, 0), (2, 3), (3, 2)]


In [33]:
# Bài 5
def count_triples_ans(nums, target):
    cnt = Counter(nums); L = len(target); total = 0
    for a in range(1, L - 1):
        for b in range(a + 1, L):
            x, y, z = target[:a], target[a:b], target[b:]
            c = cnt.copy(); ways = 1
            for part in (x, y, z):               # chọn lần lượt, mỗi lần dùng bớt một bản
                ways *= c[part]; c[part] -= 1
                if ways <= 0: ways = 0; break
            total += ways
    return total

def count_triples_brute(nums, target):
    n = len(nums)
    return sum(1 for i, j, k in permutations(range(n), 3) if nums[i] + nums[j] + nums[k] == target)

rng = random.Random(5)
for _ in range(300):
    target = "".join(rng.choice("12") for _ in range(rng.randint(3, 6)))
    nums = ["".join(rng.choice("12") for _ in range(rng.randint(1, 3))) for _ in range(rng.randint(3, 8))]
    assert count_triples_ans(nums, target) == count_triples_brute(nums, target)
print("Bài 5: OK; ['1','1','1','1'], '111' ->", count_triples_ans(["1"] * 4, "111"))

Bài 5: OK; ['1','1','1','1'], '111' -> 24


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm cặp có thứ tự ghép thành target":**

```python
cnt = Counter(nums); ans = 0
for k in range(1, len(target)):
    p, s = target[:k], target[k:]
    ans += cnt[p] * (cnt[s] - (p == s))
return ans
```

In [34]:
def template(nums, target):
    cnt = Counter(nums)
    return sum(cnt[target[:k]] * (cnt[target[k:]] - (target[:k] == target[k:])) for k in range(1, len(target)))
print([template(n, t) for n, t, _ in examples])

[4, 2, 6]


**Checklist:**

- [ ] Cặp **có thứ tự** (`(i, j)` và `(j, i)` khác nhau).
- [ ] `i ≠ j`: khi tiền tố bằng hậu tố dùng `c·(c−1)`.
- [ ] Dùng `Counter` (có lặp), không dùng `set`.
- [ ] Nếu kiểm tiền tố/hậu tố riêng, nhớ điều kiện **tổng độ dài**.
- [ ] Vị trí cắt `k` từ `1` tới `L − 1` (mỗi vế khác rỗng).
- [ ] Đã thử: ví dụ 3 (mọi chuỗi giống nhau), chuỗi dài hơn target.